# Ethiopia EDA

Task 2 implementation for Ethiopia with full profiling, cleaning, export, and visual diagnostics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.data_loader import load_country
from src.cleaning import drop_duplicates, flag_outliers, run_cleaning_pipeline
from src.eda_utils import missing_value_report
from src.visualization import monthly_temperature, monthly_precipitation, correlation_heatmap

COUNTRY = "ethiopia"
df = load_country(COUNTRY, data_dir=Path("data"))
df, dup_count = drop_duplicates(df)
print(f"Duplicates found/dropped: {dup_count}")
display(df.describe(include=[np.number]))
na_report = missing_value_report(df)
display(na_report)
print("Columns with >5% nulls:", na_report[na_report['null_pct'] > 5].index.tolist())
flagged = flag_outliers(df)
print("Outlier rows flagged (|Z| > 3):", int(flagged['is_outlier'].sum()))
clean_df, report = run_cleaning_pipeline(df, COUNTRY, data_dir=Path("data"))
print(report)
monthly_temperature(clean_df, country=COUNTRY.capitalize())
monthly_precipitation(clean_df, country=COUNTRY.capitalize())
correlation_heatmap(clean_df, country=COUNTRY.capitalize())
sns.scatterplot(data=clean_df, x="T2M", y="RH2M", alpha=0.4)
plt.title("T2M vs RH2M")
plt.show()
sns.scatterplot(data=clean_df, x="T2M_RANGE", y="WS2M", alpha=0.4)
plt.title("T2M_RANGE vs WS2M")
plt.show()
sns.histplot(clean_df["PRECTOTCORR"], bins=40, kde=True)
plt.yscale("log")
plt.title("PRECTOTCORR distribution (log y-scale)")
plt.show()
plt.figure(figsize=(8, 6))
plt.scatter(clean_df["T2M"], clean_df["RH2M"], s=np.clip(clean_df["PRECTOTCORR"] * 3, 5, 200), alpha=0.3)
plt.xlabel("T2M")
plt.ylabel("RH2M")
plt.title("Bubble chart: T2M vs RH2M (size=PRECTOTCORR)")
plt.show()
display(clean_df.corr(numeric_only=True)["T2M"].sort_values(ascending=False).head(4))
